# Insturctions to follow before executing this script

## Step 1: Create the Isolated phase2_trellis (Run in Terminal)



### Tunnel for Jupyter
ssh -f -N -L 5402:scn33-mn:8888 rahkus@login.npsf.cdac.in
### Tunnel for Gradio
ssh -f -N -L 7862:scn33-mn:7860 rahkus@login.npsf.cdac.in


In [1]:
# 1. THE FROZEN RULE: Core Config First
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Change to 0, 1, 2, or 3
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.
[SYSTEM] Notebook strictly locked to GPU: 0


In [2]:
!kill 9 2386856

/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [3]:
!nvidia-smi

Sat Aug 22 23:21:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   29C    P0             55W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
!pip install --upgrade jupyter ipywidgets

# Step 1: TRELLIS.2 (Setup)
Full C++ Compilation & DependenciesThis cell contains the complete, native compilation logic. If it detects you already built the extensions, it will safely skip to save 15 minutes. If you ever need to rebuild, change **FORCE_RECOMPILE = True**.

In [5]:
import os
import sys
import subprocess
import shutil
import importlib.util
from core_config import SYSTEM_DIRS

FORCE_RECOMPILE = False

def run_cmd(cmd, cwd=None):
    print(f"Running: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, check=True)

def is_installed(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

print("1. Verifying PyTorch 2.6.0 (CUDA 12.4)...")
run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])

print("2. Verifying ML and 3D dependencies...")
run_cmd([
    sys.executable, "-m", "pip", "install", 
    "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", 
    "ninja", "trimesh", "transformers==4.56.1", "gradio", "tensorboard", "pandas", 
    "lpips", "zstandard", "kornia", "timm", "einops", "safetensors", "fire", 
    "accelerate", "ipywidgets"
])
run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])

extensions_exist = is_installed("nvdiffrast") and is_installed("nvdiffrec") and is_installed("cumesh") and is_installed("flex_gemm") and is_installed("o_voxel")

if not FORCE_RECOMPILE and extensions_exist:
    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")
else:
    print("3. Compiling C++ Extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, o-voxel)...")
    print("   (This will take 5-15 minutes. Please do not interrupt.)")
    EXT_DIR = os.path.join(SYSTEM_DIRS['cache'], "trellis2_extensions")
    os.makedirs(EXT_DIR, exist_ok=True)

    # Nvdiffrast
    nd_dir = os.path.join(EXT_DIR, "nvdiffrast")
    if os.path.exists(nd_dir): shutil.rmtree(nd_dir)
    run_cmd(["git", "clone", "-b", "v0.4.0", "https://github.com/NVlabs/nvdiffrast.git", nd_dir])
    run_cmd([sys.executable, "-m", "pip", "install", nd_dir, "--no-build-isolation"])

    # Nvdiffrec
    nr_dir = os.path.join(EXT_DIR, "nvdiffrec")
    if os.path.exists(nr_dir): shutil.rmtree(nr_dir)
    run_cmd(["git", "clone", "-b", "renderutils", "https://github.com/JeffreyXiang/nvdiffrec.git", nr_dir])
    run_cmd([sys.executable, "-m", "pip", "install", nr_dir, "--no-build-isolation"])

    # CuMesh (Recursive)
    cm_dir = os.path.join(EXT_DIR, "CuMesh")
    if os.path.exists(cm_dir): shutil.rmtree(cm_dir)
    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/CuMesh.git", cm_dir, "--recursive"])
    run_cmd([sys.executable, "-m", "pip", "install", cm_dir, "--no-build-isolation"])

    # FlexGEMM (Recursive)
    fg_dir = os.path.join(EXT_DIR, "FlexGEMM")
    if os.path.exists(fg_dir): shutil.rmtree(fg_dir)
    run_cmd(["git", "clone", "https://github.com/JeffreyXiang/FlexGEMM.git", fg_dir, "--recursive"])
    run_cmd([sys.executable, "-m", "pip", "install", fg_dir, "--no-build-isolation"])

t2_dir = os.path.join(SYSTEM_DIRS['models'], "TRELLIS.2")
if FORCE_RECOMPILE or not os.path.exists(os.path.join(t2_dir, "setup.sh")):
    if os.path.exists(t2_dir): shutil.rmtree(t2_dir)
    print("4. Cloning Official TRELLIS.2 Repository...")
    run_cmd(["git", "clone", "-b", "main", "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"])

if FORCE_RECOMPILE or not is_installed("o_voxel"):
    print("   Installing o-voxel...")
    run_cmd([sys.executable, "-m", "pip", "install", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])

sys.path.insert(0, t2_dir)
print("Environment Ready! TRELLIS.2 mapped and ready.")

1. Verifying PyTorch 2.6.0 (CUDA 12.4)...
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
2. Verifying ML and 3D dependencies...
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja trimesh transformers==4.56.1 gradio tensorboard pandas lpips zstandard kornia timm einops safetensors fire accelerate ipywidgets
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
  Cloning https://github.com/EasternJournalist/utils3d.git (to revision 9a4eb15e4021b67b12c460c7057d642626897ec8) to ./aireos_cache/pip-req-build-okchu2bq


  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-req-build-okchu2bq
  Running command git rev-parse -q --verify 'sha^9a4eb15e4021b67b12c460c7057d642626897ec8'
  Running command git fetch -q https://github.com/EasternJournalist/utils3d.git 9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8


  Resolved https://github.com/EasternJournalist/utils3d.git to commit 9a4eb15e4021b67b12c460c7057d642626897ec8
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
3. Compiling C++ Extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, o-voxel)...
   (This will take 5-15 minutes. Please do not interrupt.)
Running: git clone -b v0.4.0 https://github.com/NVlabs/nvdiffrast.git /nlsasfs/home/lmedge/rahkus/aireos_cache/trellis2_extensions/nvdiffrast


Cloning into '/nlsasfs/home/lmedge/rahkus/aireos_cache/trellis2_extensions/nvdiffrast'...
Note: switching to '253ac4fcea7de5f396371124af597e6cc957bfae'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install /nlsasfs/home/lmedge/rahkus/aireos_cache/trellis2_extensions/nvdiffrast --no-build-isolation
Processing ./aireos_cache/trellis2_extensions/nvdiffrast
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


  error: subprocess-exited-with-error
  
  × Building wheel for nvdiffrast (pyproject.toml) did not run successfully.
  │ exit code: 2
  ╰─> [1 lines of output]
      Intel oneMKL FATAL ERROR: Cannot load /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/lib/python3.10/site-packages/torch/lib/libtorch_cpu.so.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for nvdiffrast
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> nvdiffrast


Failed to build nvdiffrast


CalledProcessError: Command '['/nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python', '-m', 'pip', 'install', '/nlsasfs/home/lmedge/rahkus/aireos_cache/trellis2_extensions/nvdiffrast', '--no-build-isolation']' returned non-zero exit status 1.

# Step 2: TRELLIS.2 (Interactive Studio - Image Upload)

In [ ]:
import core_config
from core_config import SYSTEM_DIRS

import os
import sys
import torch
import random
import gc
from PIL import Image
import gradio as gr

from purge_manager import DiskPurgeManager
from storage_manager import release_reserve, resume_reserve

os.environ["ATTN_BACKEND"] = "xformers"
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

release_reserve('models')
try:
    print("Loading TRELLIS.2-4B (Advanced 3D Extruder)...")
    trellis_pipe = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
    trellis_pipe.cuda()
finally:
    resume_reserve('models')

print("\nExecuting VRAM Load & Purge...")
DiskPurgeManager.purge_hf_model("microsoft/TRELLIS.2-4B")
DiskPurgeManager.purge_directory(os.environ["TRELLIS_CACHE_DIR"], "Trellis Cache")

@torch.inference_mode()
def extrude_3d_asset(input_image, decimation_target: int):
    if input_image is None:
        raise gr.Error("Please upload an image first!")
        
    seed = random.randint(0, 999999)
    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")
    
    input_image = input_image.convert("RGB")
    mesh = trellis_pipe.run(input_image)[0]
    mesh.simplify(16777216) 
    
    glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")
    glb = o_voxel.postprocess.to_glb(
        vertices            =   mesh.vertices,
        faces               =   mesh.faces,
        attr_volume         =   mesh.attrs,
        coords              =   mesh.coords,
        attr_layout         =   mesh.layout,
        voxel_size          =   mesh.voxel_size,
        aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target   =   int(decimation_target),
        texture_size        =   4096,
        remesh              =   True,
        remesh_band         =   1,
        remesh_project      =   0,
        verbose             =   False
    )
    glb.export(glb_path, extension_webp=True)
    
    del mesh, glb
    gc.collect()
    torch.cuda.empty_cache()
    
    return glb_path

with gr.Blocks(title="TRELLIS.2 Standalone", theme=gr.themes.Monochrome()) as demo_trellis:
    gr.Markdown("# 🕹️ TRELLIS.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload 2D Concept", type="pil")
            decimation = gr.Slider(label="Target Polygons", minimum=10000, maximum=500000, value=100000, step=10000)
            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")
        with gr.Column(scale=2):
            output_3d = gr.Model3D(label="3D Interactive Viewer")

    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])

print("Starting TRELLIS Server on port 7862...")
demo_trellis.launch(server_name="0.0.0.0", server_port=7862, inline=True, share=False)

# Step 3: FLUX.2 (Setup)
This installs the specific diffusers patch required for FLUX.2 while protecting your PyTorch 2.6.0 environment.

In [ ]:
import sys
import subprocess

print("Installing FLUX.2 Diffusers Bridge (Safe Mode)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-U", 
    "git+https://github.com/huggingface/diffusers.git", 
    "huggingface-hub<1.0", "transformers==4.56.1"
], check=True)
print("✅ FLUX.2 Dependencies installed safely!")

# Step 4: FLUX.2 (Interactive Studio - Text to Image)

In [ ]:
import core_config
from core_config import SYSTEM_DIRS

import os
import torch
import random
import gc
import gradio as gr
from diffusers import DiffusionPipeline

from purge_manager import DiskPurgeManager
from storage_manager import release_reserve, resume_reserve

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "images")
os.makedirs(OUTPUT_DIR, exist_ok=True)

release_reserve('models')
try:
    print("Loading FLUX.2 [klein] 4B (Isolated)...")
    flux_pipe = DiffusionPipeline.from_pretrained(
        "black-forest-labs/FLUX.2-klein-4B", 
        torch_dtype=torch.bfloat16,
        cache_dir=SYSTEM_DIRS['models'],
        trust_remote_code=True,
        text_encoder_2=None,
        tokenizer_2=None,
        feature_extractor=None,
        image_encoder=None
    )
    flux_pipe.enable_model_cpu_offload() 
finally:
    resume_reserve('models')

print("\nExecuting VRAM Load & Purge...")
DiskPurgeManager.purge_hf_model("black-forest-labs/FLUX.2-klein-4B")

@torch.inference_mode()
def generate_concept(prompt: str, seed: int, randomize: bool):
    if randomize: seed = random.randint(0, 999999)
    print(f"\n[GENERATING] Prompt: '{prompt}' | Seed: {seed}")
    
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = flux_pipe(
        prompt,
        guidance_scale=3.5, 
        num_inference_steps=28, 
        max_sequence_length=512,
        generator=generator
    ).images[0].convert("RGB")
    
    out_path = os.path.join(OUTPUT_DIR, f"flux2_concept_{seed}.png")
    image.save(out_path)
    
    gc.collect()
    torch.cuda.empty_cache()
    return image, seed

with gr.Blocks(title="FLUX.2 Standalone", theme=gr.themes.Monochrome()) as demo_flux:
    gr.Markdown("# 🎨 FLUX.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", value="A sci-fi crate", lines=4)
            seed = gr.Number(label="Seed", value=42, precision=0)
            randomize = gr.Checkbox(label="Randomize", value=True)
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=2):
            output_image = gr.Image(label="FLUX.2 Output")

    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize], outputs=[output_image, seed])

print("Starting FLUX Server on port 7863...")
demo_flux.launch(server_name="0.0.0.0", server_port=7863, inline=True, share=False)

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi